# Agentic RAG on Azure — health-model demo

This notebook drives a small **agentic RAG** research assistant and then deliberately
stresses each dependency so you can watch the **Azure Monitor health model** react.

**Request path the model observes**

```
researcher question
      │
      ▼
  Agent (Azure OpenAI Responses API, gpt-4.1-mini)
      │   via APIM AI gateway  ──►  Azure OpenAI backend pool (East US 2 + West US)
      ├── tool: Foundry IQ knowledge base (Azure AI Search)  — grounding on the papers
      └── tool: Scholarly-papers MCP server (Container Apps)  — citations / cross-checks
      │
      ▼
  a cited answer
```

**What the health model watches** (one entity per dependency, rolled up to a root):

| Entity | Impact | Key signals |
|---|---|---|
| APIM AI gateway | Standard | derives from the Azure OpenAI pool (WorstOf) |
| Azure OpenAI backend pool | Standard | MinHealthy over the two regions |
| Azure OpenAI — Primary / Secondary | Standard | availability, **time-to-last-byte** (static; ML-ready), 429 throttling |
| Foundry IQ knowledge base | Standard | search latency, throttled queries |
| Scholarly-papers MCP server | **Limited** | running replicas, restarts |

`Standard` impact means the dependency can make the assistant **Unhealthy**. `Limited`
impact (the MCP tool) can only make it **Degraded** — the agent still answers from the
knowledge base, so losing the tool is an enhancement loss, not an outage.

**Prerequisites** — `az login` (same tenant as the deployment), the base infra + knowledge
base + health model deployed, and `AgenticRAGHealthModeling/.env` populated. Install deps:
`pip install openai httpx azure-identity python-dotenv`.

## 1 — Config and the Responses client (through APIM)

We point the native OpenAI Responses SDK at the **APIM gateway** instead of a single
Azure OpenAI resource. The only secret the client holds is the gateway subscription key;
APIM's managed identity mints the real backend token and load-balances the two regions.

In [1]:
# WHAT: load .env and build one OpenAI Responses client that talks to the APIM AI gateway.
# GOAL: every call in this notebook flows client -> APIM -> Azure OpenAI pool, exactly the
#       path the health model observes (so the load we generate moves the real signals).
import os, time, json, statistics, concurrent.futures as cf, subprocess
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI, BadRequestError

load_dotenv(Path.cwd().parent / '.env' if (Path.cwd().name == 'infra') else Path.cwd() / '.env')

APIM_BASE = os.environ['APIM_RESPONSES_BASE_URL'].rstrip('/') + '/'   # client appends 'responses'
APIM_KEY  = os.environ['APIM_SUBSCRIPTION_KEY']
MODEL     = os.environ.get('CHAT_MODEL', 'gpt-4.1-mini')

client = OpenAI(
    api_key='apim-gateway',                                    # placeholder; APIM MI does real auth
    base_url=APIM_BASE,
    default_headers={'Ocp-Apim-Subscription-Key': APIM_KEY},
)

def text_of(resp):
    """Assistant text from a Responses result."""
    t = getattr(resp, 'output_text', None)
    if t:
        return t
    data = resp.model_dump() if hasattr(resp, 'model_dump') else resp
    out = []
    for item in data.get('output', []):
        if item.get('type') == 'message':
            for c in item.get('content', []):
                if c.get('type') == 'output_text':
                    out.append(c.get('text', ''))
    return '\n'.join(out).strip()

def trace(resp):
    """One line per output item so you can SEE the agentic steps (tool calls, message)."""
    data = resp.model_dump() if hasattr(resp, 'model_dump') else resp
    lines = []
    for item in data.get('output', []):
        t = item.get('type')
        if t == 'mcp_list_tools':
            lines.append(f"[mcp_list_tools] server={item.get('server_label')}")
        elif t == 'mcp_call':
            lines.append(f"[mcp_call] {item.get('server_label')}.{item.get('name')}")
        elif t == 'message':
            lines.append('[message]')
        else:
            lines.append(f'[{t}]')
    return '\n'.join(lines)

print('Gateway :', APIM_BASE)
print('Model   :', MODEL)
print('Ready — calls flow client -> APIM -> Azure OpenAI pool.')

Gateway : https://agrag-gateway.azure-api.net/responses-ha/openai/v1/
Model   : gpt-4.1-mini
Ready — calls flow client -> APIM -> Azure OpenAI pool.


## 2 — The agent's tools (Foundry IQ knowledge base + scholarly MCP)

Two remote **MCP** tools are attached to the Responses call:

- **`foundry_iq_kb`** — the Foundry IQ knowledge base over the ingested papers. Reached over
  MCP with a fresh Azure AI Search token minted per call.
- **`scholarly_papers`** — our MCP server on Container Apps (Semantic Scholar: references,
  citations, recommendations).

We warm + health-check each server first: a single unreachable MCP server would fail the
whole `/responses` request, so we attach only the ones that answer.

In [2]:
# WHAT: warm-check each MCP server, then build the tool list for the Responses call.
# GOAL: attach the knowledge base + scholarly tools the agent will choose between; skip any
#       server that is down so a cold/broken tool never fails the whole request.
import httpx
from azure.identity import AzureCliCredential

TENANT      = os.environ.get('AZURE_TENANT_ID')
MCP_URL     = os.environ['MCP_URL']                              # scholarly-papers MCP (Container Apps)
SEARCH      = os.environ['SEARCH_ENDPOINT'].rstrip('/')
KB_NAME     = os.environ.get('KNOWLEDGE_BASE', 'research-papers-kb')
KB_MCP_URL  = f'{SEARCH}/knowledgebases/{KB_NAME}/mcp?api-version=2026-05-01-preview'

def _search_token():
    cred = AzureCliCredential(tenant_id=TENANT, process_timeout=60) if TENANT else AzureCliCredential(process_timeout=60)
    return cred.get_token('https://search.azure.com/.default').token

def _mcp_ok(url, headers=None):
    """MCP initialize handshake — confirms reachable AND warms the server."""
    body = {'jsonrpc': '2.0', 'id': 1, 'method': 'initialize',
            'params': {'protocolVersion': '2025-06-18', 'capabilities': {},
                       'clientInfo': {'name': 'warmup', 'version': '1.0'}}}
    h = {'Content-Type': 'application/json', 'Accept': 'application/json, text/event-stream'}
    if headers:
        h.update(headers)
    for _ in range(2):
        try:
            r = httpx.post(url, headers=h, json=body, timeout=60)
            if r.status_code < 400:
                return True, r.status_code
            last = r.status_code
        except Exception as exc:
            last = type(exc).__name__
    return False, last

def build_tools(verbose=True):
    """Return Responses `tools` for only the reachable MCP servers (fresh Search token each call)."""
    tools = []
    ok, st = _mcp_ok(MCP_URL)
    if verbose:
        print(f"scholarly_papers (MCP): {'ready' if ok else 'SKIP'} ({st})")
    if ok:
        tools.append({'type': 'mcp', 'server_label': 'scholarly_papers',
                      'server_url': MCP_URL, 'require_approval': 'never'})
    try:
        tok = _search_token()
        ok, st = _mcp_ok(KB_MCP_URL, {'Authorization': f'Bearer {tok}'})
        if verbose:
            print(f"foundry_iq_kb (MCP)   : {'ready' if ok else 'SKIP'} ({st})")
        if ok:
            tools.append({'type': 'mcp', 'server_label': 'foundry_iq_kb', 'server_url': KB_MCP_URL,
                          'require_approval': 'never', 'headers': {'Authorization': f'Bearer {tok}'}})
    except Exception as exc:
        if verbose:
            print(f'foundry_iq_kb (MCP)   : SKIP (no Search token: {type(exc).__name__})')
    return tools

TOOLS = build_tools()
print('tools attached:', [t['server_label'] for t in TOOLS])

scholarly_papers (MCP): ready (200)
foundry_iq_kb (MCP)   : ready (200)
tools attached: ['scholarly_papers', 'foundry_iq_kb']


## 3 — Watch the health model (read the live rollup)

A small reader that polls the deployed health model over ARM and prints the same tree you see in
the portal Graph — so each scenario can show its state change **in the notebook**. Needs your
`az login` identity to have read access to the health model (Owner/Reader on the resource group).

In [4]:
# WHAT: read the health model's entities (state + signal values) over ARM and render the tree.
# GOAL: watch rollups change live while a scenario runs, instead of alt-tabbing to the portal.
import datetime
ARM          = 'https://management.azure.com'
SUB          = os.environ['AZURE_SUBSCRIPTION_ID']
RG           = os.environ['HEALTH_RG']
HEALTH_MODEL = os.environ.get('HEALTH_MODEL_NAME', 'agentic-rag-health')
HM_API       = '2026-09-01-preview'
_arm_cred = AzureCliCredential(tenant_id=TENANT, process_timeout=60) if TENANT else AzureCliCredential(process_timeout=60)
def _arm_token(): return _arm_cred.get_token('https://management.azure.com/.default').token

EMOJI = {'Healthy': '🟢', 'Degraded': '🟡', 'Unhealthy': '🔴', 'Unknown': '⚪', None: '⚪'}

def get_entities():
    url = f'{ARM}/subscriptions/{SUB}/resourceGroups/{RG}/providers/Microsoft.CloudHealth/healthmodels/{HEALTH_MODEL}/entities?api-version={HM_API}'
    r = httpx.get(url, headers={'Authorization': f'Bearer {_arm_token()}'}, timeout=60)
    r.raise_for_status()
    return {e['name']: e['properties'] for e in r.json()['value']}

def _signals(props):
    parts = []
    for g in (props.get('signalGroups') or {}).values():
        if not isinstance(g, dict):
            continue
        for s in g.get('signals', []) or []:
            st = s.get('status') or {}
            parts.append(f"{s.get('name')}={st.get('value')} {EMOJI.get(st.get('healthState'))}")
    return '  |  '.join(parts)

_TREE = [('agentic-research-assistant', 'Agentic Research Assistant (root)', 0),
         ('apim-gateway', 'APIM gateway', 1),
         ('aoai-backend-pool', 'AOAI backend pool (MinHealthy)', 1),
         ('aoai-primary-eastus2', 'AOAI PRIMARY (East US 2)', 2),
         ('aoai-secondary-westus', 'AOAI SECONDARY (West US)', 2),
         ('foundry-iq-search', 'Foundry IQ knowledge base', 1),
         ('mcp-server', 'Scholarly-papers MCP server', 1)]

def show_health(focus=None):
    e = get_entities()
    print(f'=== health model @ {datetime.datetime.now():%H:%M:%S} ===')
    for name, label, indent in _TREE:
        st = (e.get(name) or {}).get('healthState')
        print(f"{'   ' * indent}{EMOJI.get(st)} {label}  [{st}]")
        if focus and name in focus:
            sig = _signals(e.get(name) or {})
            if sig:
                print(f"{'   ' * indent}      {sig}")

def watch_health(minutes=8, interval=60, focus=None):
    end = time.time() + minutes * 60
    while True:
        show_health(focus=focus); print()
        if time.time() >= end:
            break
        time.sleep(interval)

print('show_health() / watch_health() ready.')
show_health()

show_health() / watch_health() ready.
=== health model @ 21:46:31 ===
🟢 Agentic Research Assistant (root)  [Healthy]
   🟢 APIM gateway  [Healthy]
   🟢 AOAI backend pool (MinHealthy)  [Healthy]
      🟢 AOAI PRIMARY (East US 2)  [Healthy]
      🟢 AOAI SECONDARY (West US)  [Healthy]
   🟢 Foundry IQ knowledge base  [Healthy]
   🟢 Scholarly-papers MCP server  [Healthy]


## 4 — Load & control helpers (drive one region, throttle on demand)

The failure scenarios drive **one region directly** (keyless, bypassing APIM's load balancer) so
only that region moves — the other stays Healthy and the `MinHealthy` pool shows **Degraded**
instead of a hard outage. `Drive` sends background load (big **output** = latency, big **input**
= throttling); `throttle()`/`restore()` reversibly shrink a deployment's TPM so requests get
**429** on demand. Needs `Cognitive Services OpenAI User` on the Azure OpenAI accounts.

In [5]:
# WHAT: keyless direct-to-region load + reversible capacity throttling for the scenarios.
# GOAL: move ONE region's signals deterministically; always restore() capacity afterward.
import threading
AOAI_API   = '2024-10-01'
DEPLOYMENT = os.environ.get('AOAI_DEPLOYMENT', 'gpt-4.1-mini')
ACCT   = {'primary': 'agrag-aoai-eastus2', 'secondary': 'agrag-aoai-westus'}
DIRECT = {'primary':   os.environ['AOAI_PRIMARY_AISVC'].rstrip('/')   + '/openai/v1/responses',
          'secondary': os.environ['AOAI_SECONDARY_AISVC'].rstrip('/') + '/openai/v1/responses'}
_aoai_cred = AzureCliCredential(tenant_id=TENANT, process_timeout=60) if TENANT else AzureCliCredential(process_timeout=60)
def _aoai_token(): return _aoai_cred.get_token('https://ai.azure.com/.default').token   # keyless

def make_prompt(n_chars):
    body = ('Attention is all you need. ' * ((n_chars // 27) + 1))[:n_chars]
    return 'Summarize the following note in three bullet points.\n\n' + body

class Drive:
    """Background keyless load at ONE region. Big input_chars -> throttling; big max_output_tokens -> latency."""
    def __init__(self, region, concurrency=4, input_chars=2000, max_output_tokens=1500):
        self.region = region; self.concurrency = concurrency; self.input_chars = input_chars
        self.max_output_tokens = max_output_tokens; self._stop = threading.Event(); self.codes = {}
    def _worker(self):
        tok = _aoai_token(); prompt = make_prompt(self.input_chars)
        with httpx.Client() as c:
            while not self._stop.is_set():
                try:
                    r = c.post(DIRECT[self.region], headers={'Authorization': f'Bearer {tok}', 'Content-Type': 'application/json'},
                               json={'model': MODEL, 'input': prompt, 'max_output_tokens': self.max_output_tokens}, timeout=180)
                    k = r.status_code
                except Exception as exc:
                    k = 429 if '429' in str(exc) else 'err'
                self.codes[k] = self.codes.get(k, 0) + 1
    def start(self):
        self._threads = [threading.Thread(target=self._worker, daemon=True) for _ in range(self.concurrency)]
        for t in self._threads: t.start()
        print(f'[drive:{self.region}] concurrency={self.concurrency} input_chars={self.input_chars} max_out={self.max_output_tokens}')
        return self
    def stop(self): self._stop.set(); time.sleep(2)
    def summary(self): print(f'[drive:{self.region}] status={dict(sorted(self.codes.items(), key=lambda x: str(x[0])))}')

# --- reversible capacity throttling (forces 429s deterministically) ---
_orig_cap = {}
def _dep_url(acc): return f'{ARM}/subscriptions/{SUB}/resourceGroups/{RG}/providers/Microsoft.CognitiveServices/accounts/{acc}/deployments/{DEPLOYMENT}?api-version={AOAI_API}'
def _sku(acc):
    r = httpx.get(_dep_url(acc), headers={'Authorization': f'Bearer {_arm_token()}'}, timeout=60); r.raise_for_status(); return r.json()['sku']
def throttle(region, capacity=1):
    acc = ACCT[region]; sku = _sku(acc); _orig_cap.setdefault(acc, sku['capacity'])
    httpx.patch(_dep_url(acc), headers={'Authorization': f'Bearer {_arm_token()}', 'Content-Type': 'application/json'},
                json={'sku': {'name': sku['name'], 'capacity': capacity}}, timeout=60).raise_for_status()
    print(f'[throttle] {acc}: capacity {_orig_cap[acc]} -> {capacity}')
def restore(region):
    acc = ACCT[region]; orig = _orig_cap.get(acc)
    if orig is None: print(f'[restore] {acc}: nothing to restore'); return
    sku = _sku(acc)
    httpx.patch(_dep_url(acc), headers={'Authorization': f'Bearer {_arm_token()}', 'Content-Type': 'application/json'},
                json={'sku': {'name': sku['name'], 'capacity': orig}}, timeout=60).raise_for_status()
    print(f'[restore] {acc}: capacity -> {orig}'); _orig_cap.pop(acc, None)
def show_capacities():
    for r_, a in ACCT.items(): print(f'  {r_} ({a}): capacity={_sku(a)["capacity"]}')

print('Drive(...) / throttle() / restore() / show_capacities() ready.')
show_capacities()

Drive(...) / throttle() / restore() / show_capacities() ready.
  primary (agrag-aoai-eastus2): capacity=50
  secondary (agrag-aoai-westus): capacity=50


## Scenario 0 — Healthy baseline

A normal research question. The agent plans, calls the knowledge base and/or the scholarly
MCP tool, and synthesizes a cited answer — the whole loop runs server-side in one call.

**Expected in the health model:** availability high, time-to-last-byte normal, all entities
**Healthy**, root **Healthy**.

In [ ]:
# WHAT: ask one grounded question and print the agent's tool-call trace + final answer.
# GOAL: prove the end-to-end agentic RAG path works before we start breaking dependencies.
def ask(question, tools=None, **kw):
    """Single agentic Responses call through APIM. Re-warms tools once if a server went cold."""
    tools = build_tools(verbose=False) if tools is None else tools
    try:
        return client.responses.create(model=MODEL, input=question, extra_body={'tools': tools}, **kw)
    except BadRequestError as exc:
        if 'external_connector' in str(exc) or 'mcp' in str(exc).lower():
            return client.responses.create(model=MODEL, input=question,
                                            extra_body={'tools': build_tools(verbose=False)}, **kw)
        raise

Q = ('According to the papers in the knowledge base, what problem does the Transformer'
     "'s self-attention mechanism solve compared to recurrence? Then use the scholarly"
     ' tool to name one influential paper that cites the original Transformer work.')
resp = ask(Q)
print(trace(resp))
print('\n' + text_of(resp))
print(); show_health()   # all entities should read Healthy at baseline

## Scenario 1 — Latency (time-to-last-byte)

**Why a few slow requests don't move the signal.** `AzureOpenAITTLTInMS` is an **average over a
5-minute window**. A handful of slow calls get diluted by the 1-minute canary probe's tiny
requests, so the average never crosses the threshold. To degrade it you need **sustained**
long-output load, aimed at **one region** so the *other* stays Healthy (otherwise both degrade
and the pool flips straight to Unhealthy instead of Degraded).

So the cell below drives **sustained** large-output requests **directly at the primary region**
(keyless, bypassing APIM's load balancing) and **polls the health model** until the signal moves.

**Signal + threshold.** Static guardrail on `AzureOpenAITTLTInMS`: **degraded > 8 s, unhealthy >
30 s** (5-minute average). Baseline is ~1 s; sustained ~1.5k-token generations push the primary's
average to ~12-20 s. (Latency is a great fit for a **dynamic/ML** threshold in production - it
learns the normal band - but that needs multi-day history, so the PoC ships the static guardrail.
See the README 'Signals, thresholds & roll-up' section.)

**Expected (give it ~5-8 min for the 5-minute average to fill):**
- 🟡 **AOAI PRIMARY** - time-to-last-byte crosses 8 s -> **Degraded**
- 🟢 **AOAI SECONDARY** - stays Healthy (canary only)
- 🟡 **pool -> root** - MinHealthy sees 1/2 healthy -> **Degraded** -> Sev2
- `availability` stays green and `throttling-429` stays 0 - a **latency** problem, not throttling.

In [ ]:
# WHAT: sustained long-OUTPUT load straight at the PRIMARY region (keyless), + a health watch.
# GOAL: raise the 5-min-average AzureOpenAITTLTInMS on ONE region past 8 s -> Degraded; pool
#       MinHealthy 1/2 -> Degraded -> root Degraded. Big output = latency (small input, no 429s).
d = Drive('primary', concurrency=3, input_chars=200, max_output_tokens=1500).start()
try:
    watch_health(minutes=12, interval=60, focus=['aoai-primary-eastus2', 'aoai-secondary-westus'])
finally:
    d.stop(); d.summary()
print('load stopped; time-to-last-byte settles back to baseline within ~5 min.')

## Scenario 2 — Throttling (429) → region Unhealthy, pool Degraded

429 depends on the deployment's tokens-per-minute cap. On a fresh GlobalStandard deployment the
cap is high, so a natural burst of small requests won't trip it (that's why a plain burst shows
`429: 0`). To make throttling **deterministic** we temporarily **shrink the primary deployment's
capacity to ~1k TPM**, then hammer it with **large** keyless requests directly — every request
now exceeds the cap and returns **HTTP 429**. The cell restores the capacity in a `finally`.

Azure OpenAI's `AzureOpenAIAvailabilityRate` is 5xx-based and does **not** count 429s, so each
region also carries a **Log Analytics 429 signal** (from the `RequestResponse` diagnostic logs):
**> 5 → Degraded, > 20 → Unhealthy**. Because the two regions roll up with **MinHealthy**, one
throttled region takes the pool to **Degraded** while the other keeps serving.

**Expected (429 signal lags ~1-2 min of log ingestion + a 5-min window):**
- 🔴 **primary** — 429 count high → Unhealthy (availability stays green — 429 ≠ 5xx)
- 🟢 **secondary** — Healthy
- 🟡 **pool → root** — MinHealthy 1/2 → Degraded → Sev2

In [ ]:
# WHAT: shrink the PRIMARY deployment's capacity, then hammer it with LARGE keyless requests so
#       it returns HTTP 429. Restore ALWAYS runs (finally).
# GOAL: drive the Log-Analytics 429 signal on ONE region past its threshold -> region Unhealthy;
#       pool MinHealthy 1/2 -> Degraded -> root Degraded. 429 != 5xx, so availability stays green.
throttle('primary', capacity=1)                        # ~1k TPM: large requests now 429
d = Drive('primary', concurrency=6, input_chars=40000, max_output_tokens=64).start()   # big INPUT = throttled
try:
    watch_health(minutes=12, interval=60, focus=['aoai-primary-eastus2', 'aoai-secondary-westus'])
finally:
    d.stop(); restore('primary'); d.summary()           # ALWAYS restore capacity
print('primary restored; the 429 signal clears over a fresh 5-min window.')

[throttle] agrag-aoai-eastus2: capacity 50 -> 1
[drive:primary] concurrency=6 input_chars=40000 max_out=64
=== health model @ 21:48:39 ===
🟢 Agentic Research Assistant (root)  [Healthy]
   🟢 APIM gateway  [Healthy]
   🟢 AOAI backend pool (MinHealthy)  [Healthy]
      🟢 AOAI PRIMARY (East US 2)  [Healthy]
            availability=100.0 🟢  |  time-to-last-byte=1201.5514 🟢  |  throttling-429=0.0 🟢
      🟢 AOAI SECONDARY (West US)  [Healthy]
            availability=100.0 🟢  |  time-to-last-byte=1411.1982 🟢  |  throttling-429=0.0 🟢
   🟢 Foundry IQ knowledge base  [Healthy]
   🟢 Scholarly-papers MCP server  [Healthy]

=== health model @ 21:49:42 ===
🟢 Agentic Research Assistant (root)  [Healthy]
   🟢 APIM gateway  [Healthy]
   🟢 AOAI backend pool (MinHealthy)  [Healthy]
      🟢 AOAI PRIMARY (East US 2)  [Healthy]
            availability=100.0 🟢  |  time-to-last-byte=1201.5514 🟢  |  throttling-429=0.0 🟢
      🟢 AOAI SECONDARY (West US)  [Healthy]
            availability=100.0 🟢  |  time-to

## Scenario 3 — Tool server down (Limited impact → only Degraded)

We scale the scholarly-papers MCP server to **zero replicas**. Its `Replicas` signal drops
below 1, so the **MCP entity goes Unhealthy** — but its **impact is `Limited`**, so the root
assistant only goes **Degraded**. The agent still answers from the knowledge base: losing the
citation tool is a graceful, partial degradation, not an outage.

This is the whole point of `impact`: not every unhealthy dependency deserves a page.

In [ ]:
# WHAT: scale the MCP container app to 0, ask the question again, then restore it.
# GOAL: show the MCP entity go Unhealthy while the ROOT stays only Degraded (Limited impact),
#       and the agent still returns a grounded answer from the knowledge base alone.
RG = os.environ['HEALTH_RG']
APP = 'agrag-scholar-mcp'
def scale(mn, mx):
    subprocess.run(['az','containerapp','update','-n',APP,'-g',RG,'--min-replicas',str(mn),
                    '--max-replicas',str(mx),'-o','none'], shell=(os.name=='nt'), check=False)

print('scaling MCP server to 0 ...')
scale(0, 0)
time.sleep(60)                       # let replicas drain and the Replicas metric report 0
resp = ask(Q)                        # same question; scholarly tool is now unreachable -> skipped
print(trace(resp))
print('\n' + text_of(resp)[:1200])
print('\nrestoring MCP server to 1 replica ...')
scale(1, 3)
print('done — MCP entity returns to Healthy after the metric recovers.')

## Scenario 4 — Retrieval degradation (Foundry IQ / Azure AI Search)

The knowledge-base entity watches **`SearchLatency`** (Degraded > 1 s, Unhealthy > 5 s) and
**`ThrottledSearchQueriesPercentage`** (Degraded > 5%, Unhealthy > 20%). On a Basic search
service a burst of concurrent retrievals raises latency and can throttle — degrading retrieval
quality even while the LLM path is perfectly healthy.

Because Search has **Standard** impact, sustained retrieval failure can drive the root to
Unhealthy: an assistant that can't ground its answers isn't doing its job. Grounding quality
is a first-class health signal here, not just infrastructure liveness.

Run the healthy question a few times in a loop to exercise retrieval and populate the Search
metrics, then review the `Foundry IQ knowledge base` entity in the portal.

In [ ]:
# WHAT: issue several retrieval-heavy questions back-to-back.
# GOAL: exercise the knowledge base so SearchLatency / throttling metrics populate the
#       Foundry IQ entity's signals (visible in the health model within a few minutes).
for i in range(5):
    r = ask('From the knowledge base only, list two concrete findings and cite the source paper.'
            f' (query {i})')
    print(f'  q{i}:', text_of(r)[:160].replace(chr(10), ' '), '...')
print('done — check the Foundry IQ knowledge base entity for latency/throttling signals.')

## Reading the health model

In the portal open **Azure Monitor → Health models →** your model (`agentic-rag-health`):

- **Entity graph** — the root `Agentic Research Assistant` with APIM, the Azure OpenAI pool
  (+ two regions), Foundry IQ, and the MCP server beneath it. Colour = current health state.
- **Signals** — click a region to see availability, time-to-last-byte (static guardrail; ML/dynamic in prod),
  and the 429 count; click the MCP server to see replicas and restarts.
- **Impact** — note how the MCP server going Unhealthy only made the root **Degraded** (Limited),
  while an Azure OpenAI region or Search going Unhealthy can make the root **Unhealthy** (Standard).
- **Alerts** — the root fires **Sev1** on Unhealthy and **Sev2** on Degraded to the action group.

Signals refresh about every minute and Log-Analytics-based signals lag 1–2 minutes, so give
each scenario a few minutes to show up. The canary probe keeps availability/latency populated
even when you're not running this notebook, so idle regions read **Healthy**, not Unknown.

## Recap

One agentic RAG request path, modeled as a small dependency graph:

1. **Healthy** — grounded, cited answers through APIM → Azure OpenAI pool + Foundry IQ + MCP.
2. **Latency** — a static time-to-last-byte guardrail (swap in a dynamic/ML band once there's baseline history).
3. **Throttling** — a Log Analytics 429 signal catches what 5xx availability misses; MinHealthy
   + APIM retry contain it to one region.
4. **Tool down** — `Limited` impact keeps a lost enhancement (the MCP tool) from paging as an outage.
5. **Retrieval** — grounding quality (Search latency/throttling) is a first-class `Standard` signal.

The model reads only platform metrics, Log Analytics, and Resource Health — it never touches
the workload. Impact and well-chosen thresholds (static here, ML/dynamic in production) are what
turn raw telemetry into a signal an on-call engineer can trust.